# SystemTopology `components_at_node` as a Derived View

Regression test for
[#635](https://github.com/sogno-platform/dpsim/issues/635):
`components_at_node` is now derived from the component list instead of
being hand-maintained state, so it cannot drift away from `components`.
Also covers [#626](https://github.com/sogno-platform/dpsim/issues/626):
a component with unconnected terminals must not crash the
`SystemTopology` constructor.

In [ ]:
import dpsimpy


def comp_names(system):
    return sorted(c.name() for c in system.components)


def map_names(system):
    return sorted(
        c.name() for comps in system.components_at_node.values() for c in comps
    )

In [ ]:
# 626: an unconnected component must not crash the constructor
n1 = dpsimpy.emt.SimNode("n1")
loose = dpsimpy.emt.ph1.Resistor("loose")
loose.set_parameters(R=1)

system = dpsimpy.SystemTopology(50, [n1], [loose])
assert comp_names(system) == ["loose"]
assert map_names(system) == []
print("constructor with an unconnected component does not crash: ok")

In [ ]:
# once the component is connected, the derived view picks it up
gnd = dpsimpy.emt.SimNode.gnd
loose.connect([n1, gnd])
assert map_names(system) == ["loose", "loose"]
print("connecting it later makes it appear in the map: ok")

In [ ]:
# components added after construction no longer drift out of the map
vs = dpsimpy.emt.ph1.VoltageSource("vs")
vs.set_parameters(V_ref=complex(10, 0), f_src=50)
vs.connect([gnd, n1])
system.add_component(vs)

assert "vs" in map_names(system)
print("component added after construction shows up in the map: ok")

In [ ]:
# reading the map repeatedly must not accumulate duplicate entries
first = map_names(system)
second = map_names(system)
assert first == second
print("repeated reads stay identical (rebuild is idempotent): ok")

In [ ]:
# removal semantics from 621/634 are preserved by the derived view
system.remove_component("loose")
assert comp_names(system) == ["vs"]
assert map_names(system) == ["vs", "vs"]
print("removed component is gone from the map: ok")